In [2]:
import pandas as pd

from config.settings import load_settings


def generate_python_schema(csv_path, output_file="generated_schema.py"):
    # Read CSV file
    df_schema = pd.read_csv(csv_path)

    # Mapping data type to Spark SQL data types
    type_mapping = {
        'double': 'DoubleType()',
        'int': 'IntegerType()',
        'integer': 'IntegerType()',
        'string': 'StringType()',
        'float': 'FloatType()',
        'long': 'LongType()',
        'boolean': 'BooleanType()',
        'timestamp': 'TimestampType()'
    }

    # Content of the generated Python file
    lines = [
        "from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType, FloatType, LongType, BooleanType, TimestampType",
        "\n# Auto-generated schema for Spark Streaming",
        "fraud_streaming_schema = StructType(["
    ]

    for _, row in df_schema.iterrows():
        field_name = row['column_name']
        data_type = type_mapping.get(row['data_type'].lower(), 'StringType()') # Default
        lines.append(f"    StructField('{field_name}', {data_type}, True),")

    lines.append("])")

    # Write .py file
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Generated successfully python file: {output_file}")


generate_python_schema("schema_event_raw.csv")

Generated successfully python file: generated_schema.py


# Generate SQL init

In [8]:
import pandas as pd

def generate_clickhouse_ddl(csv_path, database_name, table_name, output_sql):
    # Đọc file schema
    df = pd.read_csv(csv_path)

    # Mapping kiểu dữ liệu (tùy chỉnh theo nhu cầu)
    # Ví dụ: Double trong Spark -> Float64 trong ClickHouse
    type_mapping = {
        'integer': 'Int64',
        'long': 'Int64',
        'double': 'Float64',
        'float': 'Float32',
        'string': 'String',
        'timestamp': 'DateTime64(3)'
    }

    columns_sql = []
    columns_sql.append(f"    TransactionID UInt64")
    columns_sql.append(f"    prediction Nullable(Float64)")
    columns_sql.append(f"    probability Nullable(Array(Float64))")
    for _, row in df.iterrows():
        col_name = row['column_name']
        # Lấy kiểu dữ liệu từ mapping, nếu không có thì giữ nguyên
        raw_type = row['data_type']
        ch_type = type_mapping.get(raw_type, raw_type)

        # ClickHouse thường dùng Nullable cho các cột có thể thiếu dữ liệu
        columns_sql.append(f"    {col_name} Nullable({ch_type})")

    # Tạo câu lệnh CREATE TABLE hoàn chỉnh
    # Dùng engine MergeTree là phổ biến nhất cho ClickHouse
    ddl = f"CREATE DATABASE IF NOT EXISTS {database_name};\n \n"
    ddl += f"CREATE TABLE IF NOT EXISTS {database_name}.{table_name} (\n"
    ddl += ",\n".join(columns_sql)
    ddl += "\n)\nENGINE = MergeTree()\nORDER BY TransactionID;" # Thay đổi key tùy ý

    # Ghi ra file
    with open(output_sql, 'w') as f:
        f.write(ddl)

    print(f"Đã tạo file SQL tại: {output_sql}")

# Chạy thử
settings = load_settings()
generate_clickhouse_ddl('schema_clickhouse_final.csv', settings.storage.clickhouse.database, settings.storage.clickhouse.table, 'init_clickhouse.sql')

Đã tạo file SQL tại: init_clickhouse.sql
